In [8]:
import xarray as xr
import pandas as pd
from dask.diagnostics import ProgressBar
from pathlib import Path
import math

# -----------------------------
# CONFIG
# -----------------------------
%run Data_Config.ipynb
print(f"NetCDF input directory: {input_dir}")
print(f"Zarr output directory: {zarr_output_dir}")
print(f"Date range: {DATE_RANGE}")

station_batch_size = 100   # Number of stations processed at once
target_time = pd.date_range("1970-01-01", "2023-12-31", freq="h")  # example range
chunks = {"station": station_batch_size, "time": -1}  # tweak as needed

# -----------------------------
# PREPROCESS FUNCTION
# -----------------------------
def preprocess_one_file(path):
    """Open a single NetCDF file, reindex time, assign station coordinate."""
    ds = xr.open_dataset(path, chunks={"time": -1})
    if 'input_station_id' in ds:
        ds = ds.drop_vars('input_station_id')
    
    # Reindex time to common target
    ds = ds.reindex(time=target_time)
    
    # Assign station ID (adjust if your attribute key differs)
    station_id = ds.attrs.get("station_id", Path(path).stem)
    ds = ds.assign_coords(station=("station", [station_id]))
    
    return ds



# -----------------------------
# FIX CHUNK SIZES
# -----------------------------
def ensure_uniform_chunks(ds, station_chunk=100, time_chunk=10000):
    """
    Force uniform chunks for station and time, store all other dims as a single chunk.
    """
    chunk_map = {}
    for dim, size in ds.dims.items():
        if dim == "station":
            chunk_map[dim] = station_chunk
        elif dim == "time":
            chunk_map[dim] = time_chunk
        else:
            # Small or flag-like dims → store whole
            chunk_map[dim] = size
    return ds.chunk(chunk_map)


# -----------------------------
# MAIN BATCHING LOOP
# -----------------------------
files = sorted(input_dir.glob("*.nc"))
num_batches = 1 #math.ceil(len(files) / station_batch_size)

for i in range(num_batches):
    batch_files = files[i*station_batch_size : (i+1)*station_batch_size]
    print(f"Processing batch {i+1}/{num_batches} ({len(batch_files)} files)")
    
    batch_datasets = []
    for f in batch_files:
        batch_datasets.append(preprocess_one_file(f))
    
    # Concatenate stations in this batch
    batch_ds = xr.concat(batch_datasets, dim="station")
    
    # Ensure uniform chunks before writing
    batch_ds = ensure_uniform_chunks(batch_ds)

batch_ds


NetCDF input directory: /Users/joelmiller/HadISD_data/netcdf
Zarr output directory: /Users/joelmiller/HadISD_data/zarr
Date range: ('1970-01-01T00', '2023-12-31T23')
Processing batch 1/1 (100 files)


/var/folders/v1/7wypnx5x11qgv022zwx5p3g00000gn/T/ipykernel_44806/1962590019.py:47: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for dim, size in ds.dims.items():


<xarray.Dataset> Size: 42GB
Dimensions:                (station: 100, time: 473329, test: 71, flagged: 19,
                            reporting_v: 19, reporting_t: 1116, reporting_2: 2,
                            coordinate_length: 1)
Coordinates:
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-31
    longitude              (station, coordinate_length) float64 800B dask.array<chunksize=(100, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 800B dask.array<chunksize=(100, 1), meta=np.ndarray>
    elevation              (station, coordinate_length) float64 800B dask.array<chunksize=(100, 1), meta=np.ndarray>
  * station                (station) <U12 5kB '010010-99999' ... '013670-99999'
Dimensions without coordinates: test, flagged, reporting_v, reporting_t,
                                reporting_2, coordinate_length
Data variables: (12/26)
    station_id             (station) |S12 1kB dask.array<chunksize=(100,), meta=np.ndarray>
    temperatures           (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    dewpoints              (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    slp                    (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    stnlp                  (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    windspeeds             (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    ...                     ...
    cloud_base             (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    wind_gust              (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    past_sigwx1            (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    quality_control_flags  (station, time, test) float64 27GB dask.array<chunksize=(100, 10000, 71), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 7GB dask.array<chunksize=(100, 10000, 19), meta=np.ndarray>
    reporting_stats        (station, reporting_v, reporting_t, reporting_2) float64 34MB dask.array<chunksize=(100, 19, 1116, 2), meta=np.ndarray>
Attributes: (12/39)
    title:                       HadISD
    institution:                 Met Office Hadley Centre, Exeter, UK
    source:                      HadISD data product
    references:                  Dunn, 2019, Met Office Hadley Centre Technic...
    creator_name:                Robert Dunn
    creator_url:                 www.metoffice.gov.uk
    ...                          ...
    station_information:         Where station is a composite the station id ...
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    featureType:                 timeSeries
    processing_date:             08-Jan-2024
    history:                     Created by mk_netcdf_files.py \nDuplicate Mo...

In [9]:
for var in batch_ds.data_vars:
    print(f"{var}: {batch_ds[var].chunks}")

station_id: ((100,),)
temperatures: ((100,), (10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 3329))
dewpoints: ((100,), (10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 3329))
slp: ((100,), (10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 1

In [10]:
    # Write to Zarr
with ProgressBar():
    if i == 0:
        batch_ds.to_zarr(zarr_output_dir, mode="w")
    else:
        batch_ds.to_zarr(zarr_output_dir, mode="a", append_dim="station")

# Free memory
del batch_ds, batch_datasets

print("Zarr store written successfully!")


[                                        ] | 0% Completed | 50.88 sms

IOStream.flush timed out


[########################################] | 100% Completed | 466.65 s
Zarr store written successfully!


In [11]:
# Open zarr store using xarray
ds_combined = xr.open_zarr(zarr_output_dir)
ds_combined

<xarray.Dataset> Size: 42GB
Dimensions:                (station: 100, time: 473329, coordinate_length: 1,
                            flagged: 19, test: 71, reporting_v: 19,
                            reporting_t: 1116, reporting_2: 2)
Coordinates:
    elevation              (station, coordinate_length) float64 800B dask.array<chunksize=(100, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 800B dask.array<chunksize=(100, 1), meta=np.ndarray>
    longitude              (station, coordinate_length) float64 800B dask.array<chunksize=(100, 1), meta=np.ndarray>
  * station                (station) <U12 5kB '010010-99999' ... '013670-99999'
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-31
Dimensions without coordinates: coordinate_length, flagged, test, reporting_v,
                                reporting_t, reporting_2
Data variables: (12/26)
    cloud_base             (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    dewpoints              (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 7GB dask.array<chunksize=(100, 10000, 19), meta=np.ndarray>
    high_cloud_cover       (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    low_cloud_cover        (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    mid_cloud_cover        (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    ...                     ...
    stnlp                  (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    temperatures           (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    total_cloud_cover      (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    wind_gust              (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    winddirs               (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
    windspeeds             (station, time) float64 379MB dask.array<chunksize=(100, 10000), meta=np.ndarray>
Attributes: (12/39)
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    acknowledgement:             RJHD was supported by the Joint BEIS/Defra M...
    cdm_data_type:               station
    creator_email:               robert.dunn@metoffice.gov.uk
    creator_name:                Robert Dunn
    ...                          ...
    station_id:                  010010-99999
    station_information:         Where station is a composite the station id ...
    summary:                     Quality-controlled, sub-daily, station datas...
    time_coverage_end:           2023-12-31T23:00Z
    time_coverage_start:         1931-01-01T06:00Z
    title:                       HadISD

AttributeError: 'function' object has no attribute 'station'

In [14]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

In [15]:
hadisd = HadISDIndex()

In [18]:
hadisd.filesystem()

TypeError: argument of type 'NoneType' is not iterable

In [16]:
all_stations = hadisd.get_all_station_ids()
print(all_stations[:10])  # Show first 10

[]


In [25]:
hadisd = HadISDIndex(station = "sdfgd")

In [26]:
hadisd.filesystem()

{'sdfgd': PosixPath('/Users/joelmiller/HadISD_data/zarr')}